#Ingest sprints.json file
1. Read the file using spark dataframe reader API
2. Define and enforce schema(preserve the nested structure)
3. Add Metadata columns:
    - Source File
    - Ingestion Timestamp
4. Write to bronze delta table


In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

#### Step 1 -  Read the CSV file using the dataframe reader API

In [0]:
#define schema
from pyspark.sql.types import *

sprints_schema = StructType ([ 
     StructField("date", DateType()),
     StructField("raceName", StringType()),
     StructField("round", IntegerType()),
     StructField("season", IntegerType()),
     StructField("url", StringType()),
     StructField("constructorId", StringType()),
     StructField("driverId", StringType()),
     StructField("grid", IntegerType()),
     StructField("laps", IntegerType()),
     StructField("number", IntegerType()),
     StructField("points", FloatType()),
     StructField("position", IntegerType()),
     StructField("positionText", StringType()),
     StructField("status", StringType())
 ])



In [0]:
sprints_df= (
    spark.read
    .format('json')
    .schema(sprints_schema)
    .option('mode','FAILFAST')
    .option('multiLine','true')
    .load(source_file)
    )

In [0]:

display(sprints_df)

date,raceName,round,season,url,constructorId,driverId,grid,laps,number,points,position,positionText,status
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,red_bull,perez,2,17,11,8.0,1,1,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,ferrari,leclerc,1,17,16,7.0,2,2,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,red_bull,max_verstappen,3,17,1,6.0,3,3,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mercedes,russell,4,17,63,5.0,4,4,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,ferrari,sainz,5,17,55,4.0,5,5,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,aston_martin,alonso,8,17,14,3.0,6,6,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mercedes,hamilton,6,17,44,2.0,7,7,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,aston_martin,stroll,9,17,18,1.0,8,8,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,williams,albon,7,17,23,0.0,9,9,Finished
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mclaren,piastri,11,17,81,0.0,10,10,Finished


#### Step 2 - Add Metadata columns:
  - Source File
  - Ingestion Timestamp

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

In [0]:
display(sprints_final_df)

date,raceName,round,season,url,constructorId,driverId,grid,laps,number,points,position,positionText,status,ingestion_timestamp,source_file
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,red_bull,perez,2,17,11,8.0,1,1,Finished,2026-09-23T08:01:03.027605Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,ferrari,leclerc,1,17,16,7.0,2,2,Finished,2026-09-23T08:01:03.027605Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,red_bull,max_verstappen,3,17,1,6.0,3,3,Finished,2026-09-23T08:01:03.027605Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mercedes,russell,4,17,63,5.0,4,4,Finished,2026-09-23T08:01:03.027605Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,ferrari,sainz,5,17,55,4.0,5,5,Finished,2026-09-23T08:01:03.027605Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,aston_martin,alonso,8,17,14,3.0,6,6,Finished,2026-09-23T08:01:03.027605Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mercedes,hamilton,6,17,44,2.0,7,7,Finished,2026-09-23T08:01:03.027605Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,aston_martin,stroll,9,17,18,1.0,8,8,Finished,2026-09-23T08:01:03.027605Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,williams,albon,7,17,23,0.0,9,9,Finished,2026-09-23T08:01:03.027605Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mclaren,piastri,11,17,81,0.0,10,10,Finished,2026-09-23T08:01:03.027605Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json


#### Step 3 -  Write to bronze delta table

In [0]:
(
    sprints_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
SELECT season, COUNT(*) 
FROM formula1.bronze.sprints
GROUP BY season
ORDER BY season;

season,count(1)
null,7
2021,73
2022,60
2023,120
2024,120
2025,120


In [0]:
display(spark.read.table(table_name))

date,raceName,round,season,url,constructorId,driverId,grid,laps,number,points,position,positionText,status,ingestion_timestamp,source_file
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,red_bull,perez,2,17,11,8.0,1,1,Finished,2026-09-23T08:01:10.14839Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,ferrari,leclerc,1,17,16,7.0,2,2,Finished,2026-09-23T08:01:10.14839Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,red_bull,max_verstappen,3,17,1,6.0,3,3,Finished,2026-09-23T08:01:10.14839Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mercedes,russell,4,17,63,5.0,4,4,Finished,2026-09-23T08:01:10.14839Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,ferrari,sainz,5,17,55,4.0,5,5,Finished,2026-09-23T08:01:10.14839Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,aston_martin,alonso,8,17,14,3.0,6,6,Finished,2026-09-23T08:01:10.14839Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mercedes,hamilton,6,17,44,2.0,7,7,Finished,2026-09-23T08:01:10.14839Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,aston_martin,stroll,9,17,18,1.0,8,8,Finished,2026-09-23T08:01:10.14839Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,williams,albon,7,17,23,0.0,9,9,Finished,2026-09-23T08:01:10.14839Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mclaren,piastri,11,17,81,0.0,10,10,Finished,2026-09-23T08:01:10.14839Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
